# Questão 2
## 1. Motivação
A proporção de candidaturas femininas se traduz em mulheres eleitas? 

As mulheres são 52,8% do eleitorado brasileiro, segundo o TSE, mas ocupam uma fração bem menor dos cargos eletivos. A cota atua sobre as candidaturas, não sobre os resultados. Ela garante que as mulheres estejam na lista, mas não que sejam eleitas. Por isso, a pergunta anterior (quanto a proporção de candidatas cresceu) não basta. Ver mais mulheres na disputa só importa para a representação se isso resultar em mais mulheres eleitas.
Analisamos as eleições ordinárias de 2002 a 2024, comparando, por cargo e ano, a participação de mulheres entre candidatos e entre eleitos, e a chance de uma candidata se eleger em relação à de um candidato.

## 2. Hipótese
Esperamos que a proporção de mulheres eleitas cresça menos do que a de candidatas, de modo que a taxa de sucesso das candidatas fique abaixo da dos candidatos (razão F/M menor que 1). Esperamos esse efeito principalmente nos cargos proporcionais, onde a cota se aplica e onde o aumento das candidaturas pode incluir candidaturas com pouco apoio partidário. Nos cargos majoritários, onde não há cota e as vagas são poucas, esperamos poucas candidatas e poucas eleitas, e variações anuais grandes em termos percentuais.

Se a razão F/M ficar perto de 1, a hipótese é refutada: as candidatas teriam as mesmas chances que os candidatos, e a defasagem na representação viria sobretudo do número de candidaturas, e não do desempenho nas urnas.

## 3. Análise dos dados

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [2]:
df = pd.read_parquet("../data/resultados_candidato.parquet")
df.shape

(2925021, 12)

In [3]:
df["eleito"] = df["resultado"].fillna("").str.startswith("eleito")

# presidente: o resultado falta em 2006 e 2010; usa-se o mais votado do último turno
pres = df[(df["cargo"] == "presidente") & (df["tipo_eleicao"] == "eleicao ordinaria") & (df["turno"].notna())].copy()
pres = pres[pres["turno"] == pres.groupby("ano")["turno"].transform("max")]
venc = pres.loc[pres.groupby("ano")["votos"].idxmax()]
df.loc[venc.index, "eleito"] = True
print(venc[["ano", "sigla_partido", "genero", "votos", "resultado"]])

# contagem de eleitos nas eleições ordinárias
ordin = df[df["tipo_eleicao"] == "eleicao ordinaria"].copy()
alvo = ["deputado federal", "senador", "governador", "presidente"]
ordin[ordin["eleito"] & ordin["cargo"].isin(alvo)].groupby(["ano", "cargo"]).size().unstack("cargo")

          ano sigla_partido     genero     votos resultado
819699   2002            PT  masculino  52793364    eleito
664722   2006            PT  masculino  58295042      None
1838514  2010            PT   feminino  55752529      None
870989   2014            PT   feminino  54501118    eleito
1188561  2018           PSL  masculino  57797847    eleito
311571   2022            PT  masculino  60345999    eleito


cargo,deputado federal,governador,presidente,senador
ano,,,,
2002,513,27,1,54
2006,513,27,1,27
2010,513,27,1,54
2014,513,27,1,27
2018,513,27,1,53
2022,513,27,1,27


In [4]:
sen = df[(df["cargo"] == "senador") & (df["ano"] == 2018)]
print(sen["tipo_eleicao"].value_counts())

por_uf = sen[sen["eleito"]].groupby(["tipo_eleicao", "sigla_uf"]).size()
print(por_uf[por_uf != 2])          # estados com número de eleitos diferente de 2

tipo_eleicao
eleicao ordinaria                   356
senador                              13
eleicao suplementar senador - mt     11
Name: count, dtype: int64
tipo_eleicao                      sigla_uf
eleicao ordinaria                 MT          1
eleicao suplementar senador - mt  MT          1
dtype: int64


In [5]:
mt = df[(df["cargo"] == "senador") & (df["ano"] == 2018) & (df["sigla_uf"] == "MT")]
(mt.sort_values("votos", ascending=False)
   [["tipo_eleicao", "sigla_partido", "genero", "situacao", "resultado", "votos", "eleito"]]
   .head(6))

,tipo_eleicao,sigla_partido,genero,situacao,resultado,votos,eleito
2366439,eleicao ordinaria,DEM,masculino,deferido,eleito,490699,True
2346877,eleicao ordinaria,PSD,masculino,deferido,nao eleito,434972,False
2322587,eleicao suplementar senador - mt,PSD,masculino,deferido,eleito,371857,True
1864763,eleicao ordinaria,REPUBLICANOS,masculino,deferido,nao eleito,333082,False
1495810,eleicao ordinaria,PSDB,masculino,deferido,nao eleito,330430,False
481348,eleicao suplementar senador - mt,PATRIOTA,feminino,deferido,nao eleito,293362,False


In [6]:
# MT 2018: a base registra uma candidata que foi eleita na ordinária e cassada em 2019 
# como não eleita; corrige-se pelas 2 mais votadas da eleição ordinária.
mt_ord = mt[mt["tipo_eleicao"] == "eleicao ordinaria"].nlargest(2, "votos")
df.loc[mt_ord.index, "eleito"] = True

# confere: 54 eleitos em 2018, sem contar a suplementar
sen18 = df[(df["cargo"] == "senador") & (df["ano"] == 2018) & (df["tipo_eleicao"] == "eleicao ordinaria")]
print(sen18["eleito"].sum())   # esperado: 54

54


## Análise

In [7]:
base = df[df["tipo_eleicao"] == "eleicao ordinaria"].copy()
base = base[base["genero"].isin(["feminino", "masculino"])]

proporcionais = ["vereador", "deputado estadual", "deputado federal", "deputado distrital"]
majoritarios  = ["prefeito", "governador", "senador", "presidente"]
base = base[base["cargo"].isin(proporcionais + majoritarios)]

invalidas = ["indeferido", "indeferido com recurso", "renuncia", "cancelado",
             "cancelado com recurso", "cassado", "cassado com recurso",
             "falecido", "falecimento", "inelegivel", "impugnado",
             "nao conhecimento do pedido", "pedido nao conhecido",
             "pedido nao conhecido com recurso"]
base["valida"] = ~base["situacao"].isin(invalidas)

In [8]:
def pct_fem(d, chaves):
    t = d.groupby(chaves + ["genero"], observed=True).size().unstack("genero", fill_value=0)
    t["total"] = t["feminino"] + t["masculino"]
    t["pct_fem"] = t["feminino"] / t["total"] * 100
    return t.reset_index()

cand = pct_fem(base, ["ano", "cargo"])
eleit = pct_fem(base[base["eleito"]], ["ano", "cargo"])

cmp_ = cand.merge(eleit, on=["ano", "cargo"], suffixes=("_cand", "_eleit"))
cmp_["dif_pp"] = cmp_["pct_fem_eleit"] - cmp_["pct_fem_cand"]

In [9]:
v = base[base["valida"]]

t = v.groupby(["ano", "cargo", "genero"], observed=True)["eleito"].agg(["sum", "count"])
t["taxa"] = t["sum"] / t["count"] * 100

taxa = t["taxa"].unstack("genero")
taxa["razao_f_m"] = taxa["feminino"] / taxa["masculino"]
taxa = taxa.reset_index()

In [10]:
taxa[taxa["cargo"].isin(proporcionais)].round({"feminino": 1, "masculino": 1, "razao_f_m": 2})

genero,ano,cargo,feminino,masculino,razao_f_m
0,2002,deputado distrital,3.9,3.8,1.02
1,2002,deputado estadual,8.0,9.5,0.85
2,2002,deputado federal,8.7,12.7,0.69
7,2004,vereador,8.5,16.8,0.51
8,2006,deputado distrital,2.2,4.1,0.54
9,2006,deputado estadual,7.6,9.3,0.82
10,2006,deputado federal,7.2,10.8,0.66
15,2008,vereador,9.1,17.9,0.51
16,2010,deputado distrital,2.0,3.4,0.59
17,2010,deputado estadual,5.5,9.6,0.57


In [11]:
cmp_[cmp_["cargo"].isin(proporcionais)].round(1)

genero,ano,cargo,feminino_cand,masculino_cand,total_cand,pct_fem_cand,feminino_eleit,masculino_eleit,total_eleit,pct_fem_eleit,dif_pp
0,2002,deputado distrital,137,526,663,20.7,5,19,24,20.8,0.2
1,2002,deputado estadual,1686,9871,11557,14.6,129,906,1035,12.5,-2.1
2,2002,deputado federal,501,3831,4332,11.6,42,471,513,8.2,-3.4
7,2004,vereador,81074,286834,367908,22.0,6537,45186,51723,12.6,-9.4
8,2006,deputado distrital,134,511,645,20.8,3,21,24,12.5,-8.3
9,2006,deputado estadual,1722,10474,12196,14.1,121,914,1035,11.7,-2.4
10,2006,deputado federal,665,4586,5251,12.7,45,468,513,8.8,-3.9
15,2008,vereador,76846,272406,349252,22.0,6487,45290,51777,12.5,-9.5
16,2010,deputado distrital,224,660,884,25.3,4,20,24,16.7,-8.7
17,2010,deputado estadual,3246,11019,14265,22.8,134,901,1035,12.9,-9.8


## 4. Decisões sobre a análise

Só eleições ordinárias, sem as suplementares.
“Eleito” inclui as três formas (eleito, eleito por qp, eleito por media). Suplentes não contam.
Presidente em 2006 e 2010: o eleito foi identificado pelo maior número de votos no último turno, porque o resultado está ausente na base.
Em 2008, uma parte mínima das candidaturas (cerca de 0,81% dos eleitos) não tem resultado registrado e foi tratada como não eleita.
Senador de MT em 2018: os eleitos foram identificados pelo maior número de votos, pois um deles teve o mandato cassado em 2019 e a vaga foi preenchida por eleição suplementar.

## 5. Gráficos

In [12]:
pio.templates.default = "plotly_white"
 
COR_GENERO = {"Feminino": "#B5338A", "Masculino": "#2B7BB9"}
COR_SERIE = {"Candidatas": "#7F7F7F", "Eleitas": "#B5338A"}
DASH_SERIE = {"Candidatas": "dot", "Eleitas": "solid"}
SIMBOLO_SERIE = {"Candidatas": "circle-open", "Eleitas": "diamond"}

PCT_ELEITORADO = 52.8   # % de mulheres no eleitorado (TSE), citado na proposta
MIN_ELEITAS = 10        # mínimo de eleitas para calcular a razão F/M (figura 2)

def limpar_facetas(fig):
    """Troca 'cargo=Vereador' por 'Vereador' nos títulos dos painéis."""
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    return fig

In [17]:
def fig_pct(cmp, cargos, titulo, ncol=2, ymax=60, altura_painel=320):
    d = cmp[cmp["cargo"].isin(cargos)].copy()
 
    # formato longo: uma linha por (ano, cargo, série)
    longo = d.melt(
        id_vars=["ano", "cargo", "total_cand", "total_eleit"],
        value_vars=["pct_fem_cand", "pct_fem_eleit"],
        var_name="serie", value_name="pct",
    )
    longo["n_total"] = np.where(longo["serie"] == "pct_fem_cand",
                                longo["total_cand"], longo["total_eleit"])
    longo["serie"] = longo["serie"].map(
        {"pct_fem_cand": "Candidatas", "pct_fem_eleit": "Eleitas"})
    longo["cargo"] = longo["cargo"].str.title()
    longo["pct"] = longo["pct"].round(1)
 
    fig = px.line(
        longo, x="ano", y="pct",
        color="serie", symbol="serie", line_dash="serie",
        color_discrete_map=COR_SERIE,
        line_dash_map=DASH_SERIE,
        symbol_map=SIMBOLO_SERIE,
        facet_col="cargo", facet_col_wrap=ncol,
        category_orders={"cargo": [c.title() for c in cargos],
                         "serie": ["Candidatas", "Eleitas"]},
        markers=True,
        hover_data={"n_total": ":,.0f", "pct": ":.1f",
                    "serie": False, "cargo": False},
        labels={"ano": "Ano da eleição", "pct": "Mulheres (%)",
                "serie": "", "n_total": "Total (homens + mulheres)"},
        title=titulo,
    )
    limpar_facetas(fig)
 
    # cada cargo tem seus próprios anos (municipais x gerais): eixo x livre
    fig.update_xaxes(matches=None, showticklabels=True,
                 type="category", categoryorder="category ascending")
    # eixo y igual em todos os painéis, para comparar a distância entre curvas
    fig.update_yaxes(range=[0, ymax], ticksuffix="%")
 
    fig.add_hline(
        y=PCT_ELEITORADO, line_dash="dash", line_color="#444", line_width=1,
        annotation_text=f"Eleitorado feminino ({PCT_ELEITORADO:.1f}%)".replace(".", ","),
        annotation_position="top left", annotation_font_size=10,
        row="all", col="all",
    )
 
    n_linhas = int(np.ceil(len(cargos) / ncol))
    fig.update_layout(height=altura_painel * n_linhas + 100,
                      legend_title_text="")
    return fig

fig1 = fig_pct(
    cmp_, proporcionais,
    # reescrevam o título para dizer a conclusão depois de ver o gráfico
    titulo="Candidatas e eleitas em cargos proporcionais (% de mulheres)",
)
fig1.show()

In [18]:
fig1b = fig_pct(
    cmp_, ["prefeito"],
    titulo="Candidatas e eleitas a prefeito (% de mulheres)",
    ncol=1, altura_painel=420,
)
fig1b.show()

In [22]:
g = (
    base[base["valida"]]
    .groupby(["ano", "cargo", "genero"], observed=True)["eleito"]
    .agg(eleitos="sum", candidatos="count")
    .unstack("genero", fill_value=0)
)
g.columns = [f"{medida}_{gen[:1]}" for medida, gen in g.columns]  # eleitos_f, ...
g = g.reset_index()
 
g["taxa_f"] = g["eleitos_f"] / g["candidatos_f"] * 100
g["taxa_m"] = g["eleitos_m"] / g["candidatos_m"] * 100
g["razao"] = (g["taxa_f"] / g["taxa_m"]).replace([np.inf, -np.inf], np.nan)
 
razao = g[g["cargo"].isin(proporcionais)].copy()
 
# razões com poucas eleitas no numerador são instáveis: ficam fora
instaveis = razao["eleitos_f"] < MIN_ELEITAS
print(f"Pontos removidos por terem menos de {MIN_ELEITAS} eleitas: "
      f"{int(instaveis.sum())} de {len(razao)}")
razao = razao[~instaveis & razao["razao"].notna()].copy()
 
razao["cargo"] = razao["cargo"].str.title()
for c in ["taxa_f", "taxa_m", "razao"]:
    razao[c] = razao[c].round(2)
 
fig2 = px.line(
    razao, x="ano", y="razao",
    color="cargo", symbol="cargo", markers=True,
    log_y=True,
    category_orders={"cargo": [c.title() for c in proporcionais]},
    hover_data={"taxa_f": ":.1f", "taxa_m": ":.1f", "razao": ":.2f",
                "eleitos_f": True, "candidatos_f": True, "cargo": False},
    labels={"ano": "Ano da eleição",
            "razao": "Razão F/M da taxa de sucesso (escala log)",
            "cargo": "Cargo",
            "taxa_f": "Taxa mulheres (%)", "taxa_m": "Taxa homens (%)",
            "eleitos_f": "Mulheres eleitas",
            "candidatos_f": "Candidaturas femininas válidas"},
    title="Chance de uma candidata se eleger em relação a um candidato",
)
anos_eleicao = sorted(razao["ano"].unique())
fig2.update_xaxes(tickmode="array", tickvals=anos_eleicao, tickformat="d")
fig2.update_yaxes(range=[np.log10(0.2), np.log10(1.3)],
                  tickvals=[0.25, 0.5, 0.75, 1],
                  ticktext=["0,25", "0,5", "0,75", "1"])
fig2.add_hline(y=1, line_dash="dash", line_color="#444", line_width=1)
fig2.add_annotation(
    x=1, xref="x domain",          # encostado na borda direita do gráfico
    y=0, yref="y",                 # log10(1) = 0, ou seja, a altura da linha
    text="Chances iguais (razão = 1)",
    showarrow=False,
    xanchor="right", yanchor="bottom",   # texto logo acima da linha
    font=dict(size=10),
)
fig2.update_layout(height=480, legend_title_text="")
fig2.show()

Pontos removidos por terem menos de 10 eleitas: 6 de 24


In [16]:
MAJORITARIOS_FIG = ["presidente", "governador", "senador"]
m = base[base["cargo"].isin(MAJORITARIOS_FIG)]
 
 
def contagem(d, grupo):
    c = (d.groupby(["ano", "cargo", "genero"], observed=True).size()
          .unstack("genero", fill_value=0)
          .reindex(columns=["feminino", "masculino"], fill_value=0)
          .reset_index()
          .melt(id_vars=["ano", "cargo"], var_name="genero", value_name="n"))
    c["grupo"] = grupo
    return c
 
 
cont = pd.concat([contagem(m, "Candidaturas"),
                  contagem(m[m["eleito"]], "Eleitos")], ignore_index=True)
cont["genero"] = cont["genero"].map({"feminino": "Feminino",
                                     "masculino": "Masculino"})
cont["cargo"] = cont["cargo"].str.title()
cont["ano"] = cont["ano"].astype(str)   # eixo categórico: barras com largura uniforme
 
fig3 = px.bar(
    cont, x="ano", y="n", color="genero", barmode="group",
    facet_row="cargo", facet_col="grupo",
    color_discrete_map=COR_GENERO,
    category_orders={"cargo": [c.title() for c in MAJORITARIOS_FIG],
                     "grupo": ["Candidaturas", "Eleitos"],
                     "genero": ["Feminino", "Masculino"],
                     "ano": sorted(cont["ano"].unique())},
    text_auto=True,
    labels={"ano": "Ano da eleição", "n": "Número de pessoas", "genero": ""},
    title="Candidaturas e eleitos em cargos majoritários (contagens absolutas)",
)
limpar_facetas(fig3)
# cada painel com a própria escala: eleitos são poucos e sumiriam ao lado das candidaturas
fig3.update_yaxes(matches=None, showticklabels=True)
fig3.update_xaxes(showticklabels=True)
fig3.update_layout(height=900, legend_title_text="")
fig3.show()

## 6. Conclusão

A Figura 1 compara, em cada cargo proporcional, a participação de mulheres entre as candidaturas e entre os eleitos. Em nenhum cargo as mulheres passam de cerca de 36% das candidaturas, e a participação entre as eleitas é sempre menor, com a única exceção de deputado distrital em 2002, em que as duas coincidem. A proporção de candidatas dá um salto depois da obrigatoriedade da cota: para deputado federal, vai de cerca de 12% em 2006 para 22% em 2010 e 32% em 2014, e para vereador, de 22% em 2008 para 33% em 2012. A proporção de eleitas, porém, quase não se move nesse período: a de deputadas federais passa de 8% (2002) para 10% (2014) e a de vereadoras fica em torno de 13% entre 2004 e 2016. A distância entre as duas curvas, portanto, aumenta: para deputado federal, ela vai de cerca de 3 pontos percentuais em 2002 para cerca de 21 em 2014. A partir de 2018 as eleitas crescem mais do que as candidatas e a distância diminui (cerca de 17 pontos em 2022), mas segue muito maior do que antes da expansão das candidaturas. Ou seja, a cota ampliou a presença de mulheres nas listas sem produzir, no mesmo ritmo, mulheres eleitas. Em nenhum cargo as mulheres se aproximam dos 52,8% do eleitorado.

A Figura 2 compara a taxa de sucesso de candidatas e candidatos, medida pela razão entre a proporção de candidatas que se elegeram e a proporção de candidatos que se elegeram (razão F/M). Em nenhum cargo nem ano a razão chega a 1: candidatas se elegem menos do que candidatos. Entre os deputados, a razão era mais alta no início da série (cerca de 0,8 para deputado estadual e 0,7 para federal em 2002 e 2006), mas cai fortemente em 2010 e atinge o ponto mais baixo em 2014, quando fica próxima de 0,3 nos dois cargos. Isso significa que, naquele ano, uma candidata tinha cerca de um terço da chance de um candidato de se eleger. Depois de 2014, a razão se recupera de forma moderada e chega a cerca de 0,4 em 2022, ainda muito abaixo da paridade. Para vereador, a série começa em torno de 0,5 (2004 e 2008), cai para cerca de 0,3 em 2012 e 2016 e sobe para perto de 0,4 em 2024. A trajetória é parecida nos três cargos, o que indica que o padrão não é específico de uma eleição ou de um nível de governo. Esses resultados confirmam a hipótese de que a participação feminina entre as candidaturas cresce mais do que entre os eleitos. A razão F/M, porém, mede a diferença de desempenho e não a explica: ela pode refletir candidatas com menos votos, menos recursos ou candidaturas pouco competitivas, e esses fatores são examinados nas perguntas 4 e 5. 
Nota: Foram excluídos da figura os pontos com menos de 10 mulheres eleitas, porque a razão é instável com números tão pequenos. Isso removeu 6 de 24 combinações de ano e cargo.

## Limitações
Em 2018, a eleição ordinária de MT elegeu Selma Arruda e Jayme Campos para o Senado. A senadora teve o mandato cassado pelo TSE em 2019, e a vaga foi preenchida por eleição suplementar em 2020 (Carlos Fávaro). Seguindo o critério de considerar apenas eleições ordinárias, contamos como eleitos os dois candidatos escolhidos nas urnas em 2018 e excluímos a suplementar. Assim, a análise mede quem foi eleito pelo voto, não quem exerceu mandato.

In [23]:
def unir_dep_estadual(d):
    return d.assign(cargo=d["cargo"].replace({"deputado distrital": "deputado estadual"}))

base_u = unir_dep_estadual(base)

cand_u  = pct_fem(base_u, ["ano", "cargo"])
eleit_u = pct_fem(base_u[base_u["eleito"]], ["ano", "cargo"])
cmp_u = cand_u.merge(eleit_u, on=["ano", "cargo"], suffixes=("_cand", "_eleit"))

comp = (cmp_[cmp_["cargo"] == "deputado estadual"]
        .merge(cmp_u[cmp_u["cargo"] == "deputado estadual"],
               on="ano", suffixes=("_sem", "_com")))
comp["dif_cand_pp"]  = comp["pct_fem_cand_com"]  - comp["pct_fem_cand_sem"]
comp["dif_eleit_pp"] = comp["pct_fem_eleit_com"] - comp["pct_fem_eleit_sem"]
comp[["ano", "dif_cand_pp", "dif_eleit_pp"]].round(2)

genero,ano,dif_cand_pp,dif_eleit_pp
0,2002,0.33,0.19
1,2006,0.33,0.02
2,2010,0.15,0.08
3,2014,-0.04,0.22
4,2018,-0.03,0.03
5,2022,0.05,-0.03
